Rayleigh-Taylor instability
======

This notebook models the Rayleigh-Taylor instability as described in Duretz et al. (2011). It includes an additional sticky air layer, following the model outlined in Kaus et al. (2010).

**Keywords:** Stress state, Free surface

**References**
1. Kaus, B. J., Mühlhaus, H., & May, D. A. (2010). A stabilization algorithm for geodynamic numerical simulations with a free surface. Physics of the Earth and Planetary Interiors, 181(1-2), 12-20.
2. Duretz, T., May, D. A., Gerya, T. V., & Tackley, P. J. (2011). Discretization errors and free surface stabilization in the finite difference and marker‐in‐cell method for applied geodynamics: A numerical study. Geochemistry, Geophysics, Geosystems, 12(7).

<img src="./images/kaus2010RTI_ALEIB_topo.png" width="40%"> 

In [ ]:
from underworld import UWGeodynamics as GEO
from underworld import visualisation as vis

import underworld.function as fn
import math
import numpy as np
import os

In [ ]:
u = GEO.UnitRegistry

KL = 500 * u.kilometer
K_viscosity = 1e20  * u.pascal * u.second
K_density   = 3200 * u.kilogram / u.meter**3

KM = K_density * KL**3
Kt = KM/ ( KL * K_viscosity )

GEO.scaling_coefficients["[length]"] = KL
GEO.scaling_coefficients["[time]"] = Kt
GEO.scaling_coefficients["[mass]"]= KM

In [ ]:
longtest = False
model_end_time = 0.025 * u.megayears

if "UW_LONGTEST" in os.environ or longtest:
    model_end_time = 5.5 * u.megayears

Model = GEO.Model(elementRes=(50,64),
                  minCoord=(-250. * u.kilometer, -500. * u.kilometer),  
                  maxCoord=(250. * u.kilometer, 140. * u.kilometer),
                  gravity=(0.0, -9.81 * u.meter / u.second**2))

dt = 2.5*u.kiloyear
dt_str = "%.1f" %(dt.m)
checkpoint_interval = 1e2*u.kiloyear
fdir = "1_23_06_FreeSurfaceALEIB_Kaus2010_Rayleigh-Taylor_Instability_dt"+dt_str+"ka"
Model.outputDir = fdir

In [ ]:
wavelength = GEO.nd(KL)
amplitude  = GEO.nd(5*u.kilometer)
offset     = GEO.nd(-100.*u.kilometer)
k = 2. * math.pi / wavelength

coord = fn.coord()
perturbationFn = offset + amplitude*fn.math.cos(k*coord[0])
shape = fn.input()[1] < perturbationFn

airMaterial = Model.add_material(name="Air material", shape=GEO.shapes.Layer(top=Model.top, bottom=0.*u.km))
densMaterial = Model.add_material(name="Dense material", shape=GEO.shapes.Layer(top=0.*u.km, bottom=Model.bottom))
lightMaterial = Model.add_material(name="Light material", shape=shape)

In [ ]:
airMaterial.density = 0. * u.kilogram / u.meter**3
densMaterial.density  = 3300 * u.kilogram / u.metre**3
lightMaterial.density = 3200 * u.kilogram / u.metre**3

airMaterial.viscosity = 1e18 * u.pascal * u.second
densMaterial.viscosity = 1e21 * u.pascal * u.second
lightMaterial.viscosity = 1e20 * u.pascal * u.second

In [ ]:
npoints = 1000
coords = np.ndarray((npoints, 2))
coords[:, 0] = np.linspace(GEO.nd(Model.minCoord[0]), GEO.nd(Model.maxCoord[0]), npoints)
coords[:, 1] = offset + amplitude*np.cos(k*coords[:, 0])

Model.add_passive_tracers(name="interface", vertices=coords)

In [ ]:
Fig = vis.Figure(figsize=(500, 500))
Fig.Points(Model.interface_tracers, pointSize=5.0)
Fig.Points(Model.swarm, Model.materialField,pointSize=3.,colourBar=False)
Fig.Mesh(Model.mesh)
Fig.save("Fig_Kaus2010RTI_0.png")
Fig.show()

In [ ]:
Model.set_velocityBCs(left=[0., None], right=[0., None], top=[None, None], bottom=[0.,0.])
#Model.set_velocityBCs(left=[0., None], right=[0., None], top=[None, 0.], bottom=[0.,0.])

zinit = 0*u.kilometer
Model.inter_wall = Model._get_InternalwallSets(zinit)

Model._freeSurface_ALEIB = True 
Model.freeSurface = True 

In [ ]:
Model.run_for(model_end_time, checkpoint_interval=checkpoint_interval,dt= dt)

In [ ]:
Fig.save("Fig_Kaus2010RTI_1.png")
Fig.show()